# 01 — Data Quality Assessment & Business Documentation
**Project:** Olist Customer Dissatisfaction Analysis

**Role of this notebook:** This notebook validates the outputs produced by the SQL cleaning pipeline and documents the business rationale behind key data preparation decisions.
No cleaning, transformation, or feature engineering is performed here.

**Project workflow:**
```
raw CSV -> SQL staging tables -> clean_reviews -> analysis_orders -> data_clean/*.csv
                                                                          |
                                                        +-----------------+-----------------+
                                                        v                                   v
                                             Python (02, 03 notebooks)              Power BI (Power Query)
```


In [1]:
import warnings 
from sqlalchemy.exc import SAWarning 

warnings.filterwarnings("ignore", category=SAWarning) 

import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://localhost/olist_dissatisfaction?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
# Analytical tables generated by the SQL cleaning pipeline
clean_reviews = pd.read_sql("SELECT * FROM dbo.clean_reviews", engine, parse_dates=[
                        "review_creation_date", "review_answer_timestamp"])
analysis_orders = pd.read_sql("SELECT * FROM dbo.analysis_orders", engine, parse_dates=[
                        "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
customers = pd.read_sql("SELECT * FROM dbo.raw_customers", engine)
products = pd.read_sql("SELECT * FROM dbo.raw_products", engine)
sellers = pd.read_sql("SELECT * FROM dbo.raw_sellers", engine)
order_items = pd.read_sql("SELECT * FROM dbo.raw_order_items", engine)
category_trans = pd.read_sql("SELECT * FROM dbo.raw_category_translation", engine)

# Raw tables retained for before/after comparison (Section 1)
raw_orders = pd.read_sql("SELECT * FROM dbo.raw_orders", engine, parse_dates=["order_purchase_timestamp"])
raw_reviews = pd.read_sql("SELECT * FROM dbo.raw_order_reviews", engine)

print(f'clean_reviews:   {len(clean_reviews):,} rows')
print(f'analysis_orders: {len(analysis_orders):,} rows')

clean_reviews:   98,673 rows
analysis_orders: 96,478 rows


## 1. Data Quality Assessment
Confirms that the SQL cleaning pipeline resolved the two issues it was designed to address: duplicate review rows and orders outside the analytical scope.

In [2]:
# --- Before/after: duplicate reviews ---
raw_dupes = len(raw_reviews) - raw_reviews['order_id'].nunique()
clean_dupes = len(clean_reviews) - clean_reviews['order_id'].nunique()
print(f'Raw reviews: {len(raw_reviews):,} rows, {raw_dupes} duplicate order_id rows')
print(f'clean_reviews: {len(clean_reviews):,} rows, {clean_dupes} duplicate order_id rows')
assert clean_reviews['order_id'].is_unique, 'clean_reviews should be 1 row per order_id'
print('PASS: clean_reviews is deduplicated to 1 row per order_id')

Raw reviews: 99,224 rows, 551 duplicate order_id rows
clean_reviews: 98,673 rows, 0 duplicate order_id rows
PASS: clean_reviews is deduplicated to 1 row per order_id


In [3]:
# --- Before/after: order scoping ---
raw_delivered_all_time = (raw_orders['order_status'] == 'delivered').sum()
print(f'Raw delivered orders (all time, no window): {raw_delivered_all_time:,}')
print(f'analysis_orders (Sep 2016-Aug 2018, delivered only): {len(analysis_orders):,}')
print(f'Purchase date range in analysis_orders: {analysis_orders["order_purchase_timestamp"].min()} -> {analysis_orders["order_purchase_timestamp"].max()}')

assert (analysis_orders['order_status'] == 'delivered').all(), 'analysis_orders should contain delivered orders only'
assert analysis_orders['order_purchase_timestamp'].between('2016-09-01', '2018-08-31 23:59:59').all(), 'analysis_orders should be within the scoped window'
print('PASS: analysis_orders is correctly scoped')


Raw delivered orders (all time, no window): 96,478
analysis_orders (Sep 2016-Aug 2018, delivered only): 96,478
Purchase date range in analysis_orders: 2016-09-15 12:16:38 -> 2018-08-29 15:00:37
PASS: analysis_orders is correctly scoped


### 1.1 Known edge case: `delivered` orders with a missing delivery date
A small number of orders (8) are marked as `delivered` but have a missing `order_delivered_customer_date` in the original Olist dataset. These orders are retained because they still contain valid review information, while delivery-derived metrics remain NULL and are excluded only from delivery-time analyses (see `02_eda_dissatisfaction_drivers.ipynb`, Section 2).

In [4]:
missing_delivery_date = analysis_orders[analysis_orders['order_delivered_customer_date'].isna()]
print(f"Delivered orders with a missing delivery date: {len(missing_delivery_date)} ({len(missing_delivery_date)/len(analysis_orders)*100:.3f}% of analysis_orders)")
print(f"Of these, orders with a valid review score: {missing_delivery_date['review_score'].notna().sum()}")
missing_delivery_date[['order_id','order_status','order_purchase_timestamp','order_delivered_customer_date','review_score']]


Delivered orders with a missing delivery date: 8 (0.008% of analysis_orders)
Of these, orders with a valid review score: 8


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,review_score
5032,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT,5.0
12401,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT,5.0
16859,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT,5.0
17031,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,5.0
17500,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT,5.0
64426,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT,1.0
87042,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT,5.0
92646,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT,5.0


## 2. Dataset Structure Validation
Confirms that dataset grain and table relationships remain valid after the SQL cleaning pipeline.

In [5]:
print('customer_id unique?', customers['customer_id'].is_unique)
print('customer_unique_id unique?', customers['customer_unique_id'].is_unique)
print('customer_id rows:', customers['customer_id'].nunique(), '| true customers (customer_unique_id):', customers['customer_unique_id'].nunique())
print()
print('avg items per order:', order_items.groupby('order_id').size().mean().round(2))


customer_id unique? True
customer_unique_id unique? False
customer_id rows: 99441 | true customers (customer_unique_id): 96096

avg items per order: 1.14


In [6]:
checks = {
    'order_items -> analysis_orders': order_items['order_id'].isin(analysis_orders['order_id']).mean(),
    'order_items -> products':        order_items['product_id'].isin(products['product_id']).mean(),
    'order_items -> sellers':         order_items['seller_id'].isin(sellers['seller_id']).mean(),
    'analysis_orders -> customers':   analysis_orders['customer_id'].isin(customers['customer_id']).mean(),
    'products -> category_trans':     (products['product_category_name'].isin(category_trans['product_category_name']) |
                                        products['product_category_name'].isna()).mean(),
}
for k, v in checks.items():
    print(f'{k:38s} {v*100:.2f}% resolve')
# Note: order_items -> analysis_orders resolves to <100% by design because
# order_items contains line items from orders outside the analytical scope
# (e.g. canceled or out-of-window orders). This does not indicate a referential integrity issue.

order_items -> analysis_orders         97.82% resolve
order_items -> products                100.00% resolve
order_items -> sellers                 100.00% resolve
analysis_orders -> customers           100.00% resolve
products -> category_trans             99.96% resolve


## 3. Why latest review?
`raw_order_reviews` is not guaranteed 1 row per `order_id` — some orders received more than one review submission. The SQL pipeline keeps the review with the latest `review_answer_timestamp` per order.

**Reasoning:** the most recently answered review reflects the buyer's final, settled sentiment. An earlier review row for the same order is treated as superseded rather than an independent data point (retaining both reviews would double-count a single customer's experience and artificially inflate review-based metrics).


In [7]:
print(f'{raw_dupes} orders had more than one review submission in the raw export ({raw_dupes/raw_reviews["order_id"].nunique()*100:.2f}% of reviewed orders).')
print('clean_reviews resolves each to a single row -> safe to use as the review-side join key for any 1:1 order-level metric.')


551 orders had more than one review submission in the raw export (0.56% of reviewed orders).
clean_reviews resolves each to a single row -> safe to use as the review-side join key for any 1:1 order-level metric.


## 4. Why the Sep 2016 – Aug 2018 analysis window?
The raw `order_purchase_timestamp` range extends into October 2018, but review and delivery-date coverage collapses in the final two raw months — most likely because those orders' lifecycles (delivery, review request/response) were still in progress when the dataset was extracted. Including them would understate delivery times and review coverage for no analytical benefit, so they're excluded at the SQL layer (`analysis_orders`) rather than filtered inconsistently downstream.


In [8]:
print('Full raw purchase timestamp range:', raw_orders['order_purchase_timestamp'].min(), '->', raw_orders['order_purchase_timestamp'].max())

tail = raw_orders[raw_orders['order_purchase_timestamp'] >= '2018-09-01']
tail_with_review = tail.merge(raw_reviews[['order_id','review_score']], on='order_id', how='left')
print(f'Sep-Oct 2018 raw orders: {len(tail):,} | with a review: {tail_with_review["review_score"].notna().sum():,}',
      f'({tail_with_review["review_score"].notna().mean()*100:.1f}% coverage)')

in_window_with_review = analysis_orders.merge(clean_reviews[['order_id']], on='order_id', how='inner')
print(f'In-window (Sep16-Aug18) delivered orders: {len(analysis_orders):,} | with a review: {len(in_window_with_review):,}',
      f'({len(in_window_with_review)/len(analysis_orders)*100:.1f}% coverage)')


Full raw purchase timestamp range: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
Sep-Oct 2018 raw orders: 20 | with a review: 19 (95.0% coverage)
In-window (Sep16-Aug18) delivered orders: 96,478 | with a review: 95,832 (99.3% coverage)


## Summary
| Check | Result |
|---|---|
| Duplicate review rows | 551 orders had >1 review submission; resolved in SQL by keeping the latest per `order_id` |
| Order scoping | Raw data extends to Oct 2018; review coverage collapses in the final two months → window fixed to Sep 2016–Aug 2018 at the SQL layer |
| Delivered-only filter | Applied in `analysis_orders` — non-delivered statuses have no `order_delivered_customer_date` |
| Customer grain | `customer_unique_id` ≠ `customer_id`; segmentation logic downstream must key off `customer_unique_id` |
| Referential integrity | All fact→dimension joins resolve ≥99.9% (order_items → analysis_orders is intentionally partial — see note above) |

**Next notebook:** `02_eda_dissatisfaction_drivers.ipynb`